# 3-D gamma dose comparison in pyCERR

The **gamma index** (Low et al., Med. Phys. 1998) compares two dose
distributions using a combined **dose-difference (DD)** and
**distance-to-agreement (DTA)** criterion: for every reference voxel,

> gamma = min sqrt( (distance / DTA)^2 + (doseDiff / DD)^2 )

so **gamma <= 1 passes**. This notebook uses `cerr.gamma` (a port of MATLAB
CERR's `Dose -> Gamma 3D`) to compare two doses in a `planC` and report the
**pass rate overall and per structure** — e.g. a **clinical TPS dose (Eclipse
RTDOSE)** vs a **pyCERR-computed dose**, or two pyCERR engines (pyRadPlan vs
QIB) against each other.

Edit the **CONFIG** cell (placeholders marked `# <-- EDIT`), then Run-All.
**No PHI** ships with this notebook — supply your own de-identified DICOM.

The same computation is available interactively in the pyCERR-Qt viewer under
**Tools → Gamma 3D**.

## 1. CONFIG

In [ ]:
# ----------------------------- CONFIG ------------------------------------
# A DICOM folder holding the CT (+ RTSTRUCT) and TWO RTDOSE series to compare,
# e.g. a clinical Eclipse dose and a pyCERR/re-planned dose exported to DICOM.
DICOM_DIR   = r"C:/path/to/your/DICOM/folder"   # <-- EDIT: de-identified CT + RTSTRUCT + 2x RTDOSE
SCAN_NUM    = 0            # <-- EDIT: planning CT index in planC.scan
REF_DOSE_NUM  = 0         # <-- EDIT: reference dose index (e.g. clinical Eclipse)
EVAL_DOSE_NUM = 1         # <-- EDIT: evaluated dose index (e.g. pyCERR)

# Structures to tabulate the pass rate for (names as they appear in RTSTRUCT).
STRUCT_NAMES = []         # <-- EDIT, e.g. ["GTV", "PTV", "SpinalCord"]; [] = all

# --- Gamma criteria ----------------------------------------------------------
DIST_AGREEMENT_MM = 3.0   # DTA (mm)
DOSE_AGREEMENT_PCT = 3.0  # DD (%)
THRESHOLD_PCT      = 20.0 # exclude reference voxels below this % of max
NORMALIZATION      = "global"   # 'global' or 'local'
DIST_SAMPLE_RATE   = 3    # search samples per DTA (higher = finer, slower)
# -------------------------------------------------------------------------

## 2. Load the DICOM dataset

`loadDcmDir` reads the CT, RTSTRUCT and both RTDOSE series into one `planC`.

In [ ]:
import numpy as np
from cerr import plan_container as pc
from cerr import gamma

planC = pc.loadDcmDir(DICOM_DIR)
print("scans:", [(i, s.scanType) for i, s in enumerate(planC.scan)])
print("doses:", [(i, getattr(d, 'fractionGroupID', 'dose')) for i, d in enumerate(planC.dose)])
print("structures:", [s.structureName for s in planC.structure])


def sidx(name):
    hits = [i for i, s in enumerate(planC.structure) if s.structureName == name]
    if not hits:
        raise ValueError("structure %r not found; available: %s"
                         % (name, [s.structureName for s in planC.structure]))
    return hits[0]


structNums = ([sidx(nm) for nm in STRUCT_NAMES] if STRUCT_NAMES
              else list(range(len(planC.structure))))

## 3. Compute the 3-D gamma

`gammaDose3dForScan` resamples both doses onto the scan grid (so the gamma map
aligns with the CT and its structures) and returns the gamma map plus the
overall pass rate. The evaluated dose may be on a different grid/resolution
than the reference — it is interpolated automatically.

In [ ]:
gamma3M, passRate = gamma.gammaDose3dForScan(
    REF_DOSE_NUM, EVAL_DOSE_NUM, SCAN_NUM, planC,
    distAgreement=DIST_AGREEMENT_MM,
    doseAgreement=DOSE_AGREEMENT_PCT,
    thresholdFraction=THRESHOLD_PCT / 100.0,
    doseAgreementType=NORMALIZATION,
    distSampleRate=DIST_SAMPLE_RATE)

print("gamma %.1f%%/%.1fmm (%s), threshold %.0f%%"
      % (DOSE_AGREEMENT_PCT, DIST_AGREEMENT_MM, NORMALIZATION, THRESHOLD_PCT))
print("overall pass rate: %.1f%%" % (100.0 * passRate))

## 4. Pass rate per structure

`gammaByStructure` tabulates the pass rate within each ROI — the per-structure
view shown by CERR's Gamma 3D tool.

In [ ]:
rows = gamma.gammaByStructure(gamma3M, structNums, planC)
print("%-24s %10s %12s" % ("structure", "# voxels", "pass rate"))
print("-" * 48)
for r in rows:
    pr = "n/a" if r["passRate"] != r["passRate"] else "%.1f%%" % (100 * r["passRate"])
    print("%-24s %10d %12s" % (r["structureName"][:24], r["numEvaluated"], pr))

## 5. Visualize the gamma distribution

A gamma colourwash on the CT slice with the most evaluated voxels (γ>1 in red),
and the gamma histogram.

In [ ]:
import matplotlib.pyplot as plt

scan3M = planC.scan[SCAN_NUM].getScanArray()
evaluated = ~np.isnan(gamma3M)
k = int(np.argmax(evaluated.sum(axis=(0, 1))))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(scan3M[:, :, k], cmap="gray")
gm = np.ma.masked_invalid(gamma3M[:, :, k])
im = ax1.imshow(gm, cmap="jet", alpha=0.6, vmin=0, vmax=2)
plt.colorbar(im, ax=ax1, label="gamma", fraction=0.046)
# outline failing voxels
ax1.contour(np.nan_to_num(gamma3M[:, :, k], nan=0) > 1, levels=[0.5],
            colors="red", linewidths=0.6)
ax1.set_title("gamma map, slice %d  (pass %.1f%%)" % (k, 100 * passRate))
ax1.axis("off")

gvals = gamma3M[evaluated]
ax2.hist(gvals, bins=60, range=(0, 3), color="tab:blue", alpha=0.8)
ax2.axvline(1.0, color="red", ls="--", label="gamma = 1")
ax2.set_xlabel("gamma"); ax2.set_ylabel("# voxels")
ax2.set_title("gamma histogram"); ax2.legend()
plt.tight_layout()
plt.show()

## Notes

- **Reference vs evaluated.** Gamma is asymmetric: the *reference* dose defines
  the evaluated points, the threshold, and (for `global`) the normalization
  (its maximum). Put the clinical/TPS dose as the reference for a TPS-vs-pyCERR
  comparison.
- **Global vs local.** `global` normalizes DD to the reference maximum; `local`
  normalizes DD to each voxel's dose (stricter in low-dose regions).
- **Dose-only comparison.** If you just have two dose objects and don't need a
  scan grid, use `gamma.gammaDose3dForDoses(refDoseNum, evalDoseNum, planC, ...)`
  which computes on the reference dose grid.
- **Performance.** The search is over a neighborhood of radius `2*DTA` sampled
  `DIST_SAMPLE_RATE` times per DTA; raise the rate for accuracy, lower it (or
  the grid resolution) for speed.
- **Interactive GUI.** The same tool is in the pyCERR-Qt viewer:
  **Tools → Gamma 3D (dose comparison)** — pick reference/evaluated doses, set
  the criteria, see per-structure pass rates, and overlay the gamma map / fail
  region on the views.
- **No PHI** — point `DICOM_DIR` at your own de-identified data.